# Sparse Latent Dynamics for Financial Forecasting under Market Shifts

**CSC 413 course project — reproducible experiment notebook**
Zhijie Jiang · Boyu Song · Shuo Liu · Yanming Zhao

This notebook implements the full pipeline described in the proposal, end to end:

| § | Step | Proposal section |
|---|------|------------------|
| 1 | Download SPY / QQQ / TLT / GLD daily data with `yfinance` (ported from `scripts/eda.py`) and reproduce the exploratory figures | §3 Data |
| 2 | Causal features, 40-day windows, chronological train / val / test splits, train-only scaling | §3 |
| 3 | The three models on a **shared GRU encoder**: `GRU`, `GRU+MLP`, `GRU+SINDy` | §4 Model architecture |
| 4 | Joint objective `L_pred + λ_d·L_cons + λ_s‖Ξ‖₁ (+ λ_r·L_rec)` and the training function (AdamW, early stopping on validation MSE) | §4 |
| 5 | Train the three models, 3 seeds each. `λ_d` / `λ_s` are **on/off switches** (0 or 1), not tuned: `GRU` (0, 0), `GRU+MLP` (`λ_d=1`), `GRU+SINDy` (`λ_d=1`, `λ_s=1`) | §4–§5 |
| 6 | MSE / MAE / directional accuracy, calm vs. stressed strata, zero-return baseline | §5 |
| 7 | SINDy diagnostics: retained terms, one-step latent transition error, term stability with a frozen encoder | §5 |

**Splits (proposal §3):** train 2010–2019, validation 2020–2021, held-out test 2022–2025.
Nothing that involves future observations is used as an *input*; the next-day window `X(t+1)` is used only as a training target for the dynamics-consistency term.

**How to run:** `Runtime ▸ Run all`. On a free Colab CPU runtime the whole notebook takes roughly 5–15 minutes (3 models × 3 seeds = 9 runs of a ~17k-parameter model, ≤ 80 epochs each with early stopping). Set `CFG.quick = True` in section 0.2 for a sub-minute smoke test.

---
## 0. Environment

Colab already provides PyTorch. `yfinance`, `matplotlib` and `tqdm` are installed on demand, so this cell is safe to re-run.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 0.1  Imports, device, reproducibility helpers
# ═══════════════════════════════════════════════════════════════════════════
import importlib
import importlib.util
import os
import subprocess
import sys


def ensure_packages(pkgs):
    """Import every (module, pip-name) pair, pip-installing the missing ones."""
    missing = [pip_name for imp_name, pip_name in pkgs
               if importlib.util.find_spec(imp_name) is None]
    if missing:
        print("installing:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()


ensure_packages([("yfinance", "yfinance"),
                 ("matplotlib", "matplotlib"),
                 ("tqdm", "tqdm")])

import copy
import itertools
import json
import math
import random
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib
import matplotlib.pyplot as plt
import yfinance as yf
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

try:                      # pretty tables in Jupyter / Colab
    from IPython.display import display
except Exception:         # plain python fallback
    display = print

# keep the notebook's inline backend, but stay headless-safe outside Jupyter
if "inline" not in matplotlib.get_backend().lower():
    matplotlib.use("Agg")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    """Seed everything that influences a training run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"torch {torch.__version__} | device: {DEVICE}")
print(f"numpy {np.__version__} | pandas {pd.__version__} | matplotlib {matplotlib.__version__}")

---
## 0.2 Configuration

Every hyper-parameter lives here. Defaults follow the proposal: `L = 40`, `m = 4`, 64 GRU units, batch 64, AdamW, early stopping on validation MSE.

**SINDy library.** `Θ(z)` contains a constant and *all* monomials of total degree `≤ poly_order`. With `poly_order = 6` and `m = 4` this is `C(4+6, 6) = 210` candidate terms (set `poly_order = 2` to reproduce the 15-term quadratic library of the proposal figure). Because the encoder output is `z = tanh(·)`, every monomial is bounded by 1, so the library stays numerically well behaved.

**The two loss weights are switches, not hyper-parameters.** `λ_d` and `λ_s` only take **0 or 1**: they turn a term of the objective on or off, and that is what distinguishes the three models.

| model | transition | `λ_d` (dynamics consistency) | `λ_s` (ℓ1 on `Ξ`) |
|---|---|---|---|
| `GRU` | none (`ẑ = z`) | 0 | 0 |
| `GRU+MLP` | dense MLP `F_φ` | **1** | 0 |
| `GRU+SINDy` | sparse `Θ(z)Ξ` | **1** | **1** |

**Scaling of the two terms.** So that a weight of exactly 1 is meaningful, the consistency term is a per-sample mean of `‖·‖²` and the sparsity term is a per-coefficient mean `mean(|Ξ|)`; the literal `Σ|Ξ|` of the proposal's eq. (3) is the same code with the `numel()` division removed (then `λ_s` would need to be ~1e-3, since `Σ|Ξ|` is ~1e3 times larger than `mean(|Ξ|)`).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 0.2  Configuration
# ═══════════════════════════════════════════════════════════════════════════
@dataclass
class Config:
    # ---- data (proposal §3) -------------------------------------------------
    tickers: Tuple[str, ...] = ("SPY", "QQQ", "TLT", "GLD")
    start: str = "2010-01-01"
    end: str = "2026-01-01"
    train: Tuple[str, str] = ("2010-01-01", "2019-12-31")
    val: Tuple[str, str] = ("2020-01-01", "2021-12-31")
    test: Tuple[str, str] = ("2022-01-01", "2025-12-31")
    force_download: bool = False     # True -> ignore the local CSV cache
    # ---- features -----------------------------------------------------------
    lookback: int = 40               # L
    vol_win: int = 20                # trailing realised-volatility window
    mom_win: int = 20                # trailing momentum window
    rvol_win: int = 20               # relative-volume window
    clip_z: float = 5.0              # hard cap on training-fitted |z| clip
    # ---- model (proposal §4) -----------------------------------------------
    latent_dim: int = 4              # m
    gru_hidden: int = 64
    gru_layers: int = 1
    head_hidden: int = 0             # 0 -> linear forecast head h_psi
    transition_hidden: int = 32      # width of F_phi (GRU+MLP only)
    poly_order: int = 6              # SINDy library order (<= 6 by request)
    detach_next: bool = True         # treat E(X_{t+1}) as a target in L_cons
    # ---- optimisation (proposal §4) ----------------------------------------
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    epochs: int = 80
    patience: int = 12               # early stopping on validation MSE
    lambda_r: float = 0.0            # optional L_rec weight (0 -> disabled)
    # ---- experiment ---------------------------------------------------------
    seeds: Tuple[int, ...] = (0, 1, 2)
    quick: bool = False              # True -> sub-minute smoke run


CFG = Config()

# ---- the ONLY thing that differs between the three models ---
# lambda_d and lambda_s are switches (0 = term off, 1 = term on).
LAMBDAS: Dict[str, Dict[str, float]] = {
    "GRU":       dict(lambda_d=0.0, lambda_s=0.0),   # blank control
    "GRU+MLP":   dict(lambda_d=1.0, lambda_s=0.0),   # dense transition
    "GRU+SINDy": dict(lambda_d=1.0, lambda_s=1.0),   # sparse transition
}
MODE_OF = {"GRU": "direct", "GRU+MLP": "dense", "GRU+SINDy": "sindy"}
MODELS = list(LAMBDAS)
COLORS = {"GRU": "#1f77b4", "GRU+MLP": "#ff7f0e", "GRU+SINDy": "#d62728"}

if CFG.quick:                        # fast smoke configuration
    CFG.epochs, CFG.patience = 4, 4
    CFG.seeds, CFG.poly_order = (0,), 2

n_lib = math.comb(CFG.latent_dim + CFG.poly_order, CFG.poly_order)
print(f"lookback L={CFG.lookback} | latent m={CFG.latent_dim} | GRU hidden={CFG.gru_hidden}"
      f" | batch={CFG.batch_size}")
print(f"SINDy library: |Theta(z)| = C(m+deg, deg) = {n_lib} terms, degree <= {CFG.poly_order}"
      f"  (degree 2 -> the proposal's 15 terms)")
print(f"training runs: {len(MODELS)} models x {len(CFG.seeds)} seeds = "
      f"{len(MODELS) * len(CFG.seeds)}")
print("\nmodel      transition  lambda_d  lambda_s")
for m in MODELS:
    print(f"{m:10s} {MODE_OF[m]:10s} {LAMBDAS[m]['lambda_d']:8.0f} {LAMBDAS[m]['lambda_s']:9.0f}")

---
## 1. Data

### 1.1 Download (ported from `scripts/eda.py`)

Adjusted daily closes and volumes for SPY, QQQ, TLT and GLD from Yahoo Finance via `yfinance`, 2010-01-01 → 2025-12-31. The panel is cached to `data/etf_daily.csv` exactly in the format `scripts/eda.py` writes, so the notebook and the repo script stay interchangeable (delete the CSV, or set `CFG.force_download = True`, to re-download).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.1  Download / cache the ETF panel  (port of scripts/eda.py)
# ═══════════════════════════════════════════════════════════════════════════
DATA_DIR = Path("data")
FIG_DIR = Path("figs")
RES_DIR = Path("results")
for _d in (DATA_DIR, FIG_DIR, RES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "etf_daily.csv"
TICKERS = list(CFG.tickers)


def download_etf_panel(tickers, start, end, retries=3):
    """Adjusted close + volume frames from Yahoo Finance (via yfinance)."""
    last_err = None
    raw = None
    for attempt in range(retries):
        try:
            raw = yf.download(tickers, start=start, end=end,
                              auto_adjust=True, progress=False, threads=False)
        except Exception as err:                     # transient network error
            last_err = err
            raw = None
        if raw is not None and len(raw) > 0:
            break
        print(f"  download attempt {attempt + 1}/{retries} returned nothing, retrying ...")
        time.sleep(3)
    if raw is None or len(raw) == 0:
        raise RuntimeError("yfinance returned no rows -- check the network / tickers"
                           + (f" (last error: {last_err})" if last_err else ""))

    if isinstance(raw.columns, pd.MultiIndex):
        # yfinance normally returns (field, ticker); normalise the other order too
        if set(tickers) & set(raw.columns.get_level_values(0)):
            raw = raw.swaplevel(0, 1, axis=1).sort_index(axis=1)
        available = set(raw.columns.get_level_values(1))
        missing = [t for t in tickers if t not in available]
        if missing:
            raise ValueError(f"no downloaded data for {missing}")
        close = raw["Close"][tickers].copy()
        volume = raw["Volume"][tickers].copy()
    else:                                   # single-ticker fallback
        close = raw[["Close"]].rename(columns={"Close": tickers[0]})
        volume = raw[["Volume"]].rename(columns={"Volume": tickers[0]})
    return close.astype(float), volume.astype(float)


def load_or_download(cfg, force=False):
    if CSV_PATH.exists() and not force:
        panel = pd.read_csv(CSV_PATH, header=[0, 1], index_col=0, parse_dates=True)
        close = panel["close"][list(cfg.tickers)]
        volume = panel["volume"][list(cfg.tickers)]
        print(f"loaded cached panel: {CSV_PATH}")
    else:
        close, volume = download_etf_panel(list(cfg.tickers), cfg.start, cfg.end)
        pd.concat({"close": close, "volume": volume}, axis=1).to_csv(CSV_PATH)
        print(f"downloaded panel from Yahoo Finance -> {CSV_PATH}")
    close = close.sort_index().dropna(how="any")
    volume = volume.sort_index().reindex(close.index).dropna(how="any")
    close = close.loc[volume.index]
    return close, volume


t_dl = time.time()
close, volume = load_or_download(CFG, force=CFG.force_download)
print(f"{len(close)} trading days | {close.index[0].date()} .. {close.index[-1].date()}"
      f" | {time.time() - t_dl:.1f}s")
print("missing closes:", close.isna().sum().to_dict())
display(close.tail(3).round(2))

### 1.2 Exploratory statistics and figures (proposal §3)

Same computations as `scripts/eda.py`: return moments and autocorrelations, the SPY 20-day realised-volatility stress indicator with an 80th-percentile **training-period** threshold (reported as 17.4 % in the proposal), calm-versus-stress correlations, and figures 1–4 of the proposal.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.2  Descriptive statistics  (port of scripts/eda.py)
# ═══════════════════════════════════════════════════════════════════════════
SPLITS = {k: (getattr(CFG, k)[0][:4], getattr(CFG, k)[1][:4])
          for k in ("train", "val", "test")}
L_ED = CFG.lookback

RET = np.log(close[TICKERS]).diff().dropna()
missing = close.isna().sum()

# Stress indicator: SPY 20-day realised vol (annualised) above the 80th
# percentile of the TRAIN period.
vol20 = RET["SPY"].rolling(CFG.vol_win).std() * np.sqrt(252)
thr = vol20.loc[SPLITS["train"][0]:SPLITS["train"][1]].quantile(0.8)
stress = vol20 > thr
stress_labeled = vol20.notna()

lines = [f"Data: Yahoo Finance via yfinance, adjusted close, "
         f"{RET.index[0].date()} to {RET.index[-1].date()}",
         f"Trading days with returns: {len(RET)}; missing closes per ticker: "
         f"{missing.to_dict()}",
         f"Stress threshold (SPY 20d vol, train 80th pct): {thr:.3f} annualized", ""]
lines.append("split  years      days  samples(L=40)  stress_days  stress_frac")
for name, (a, b) in SPLITS.items():
    n = len(RET.loc[a:b])
    s = int(stress.loc[a:b].sum())
    lines.append(f"{name:5s}  {a}-{b}  {n:5d}  {n - L_ED + 1:12d}  {s:11d}  {s / n:10.2%}")

desc = pd.DataFrame({
    "mean_daily_%": RET.mean() * 100,
    "std_daily_%": RET.std() * 100,
    "ann_vol_%": RET.std() * np.sqrt(252) * 100,
    "skew": RET.skew(),
    "excess_kurt": RET.kurt(),
    "min_%": RET.min() * 100,
    "max_%": RET.max() * 100,
    "lag1_autocorr": [RET[c].autocorr(1) for c in TICKERS],
    "lag1_autocorr_|r|": [RET[c].abs().autocorr(1) for c in TICKERS],
}).round(3)
lines += ["", "Daily log-return statistics (full sample):", desc.to_string()]

calm_corr = RET[~stress.reindex(RET.index, fill_value=False)].corr().round(2)
stress_corr = RET[stress.reindex(RET.index, fill_value=False)].corr().round(2)
lines += ["", "Correlation, calm days:", calm_corr.to_string(),
          "", "Correlation, stress days:", stress_corr.to_string()]

(DATA_DIR / "summary_stats.txt").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.3  Figures 1-2 of the proposal (prices + splits, vol and stress)
# ═══════════════════════════════════════════════════════════════════════════
plt.rcParams.update({"font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False})
PALETTE = {"SPY": "#1f77b4", "QQQ": "#ff7f0e", "TLT": "#2ca02c", "GLD": "#d62728"}
SPLIT_SHADE = {"val": "#dddddd", "test": "#bbbbbb"}

# ---- Fig 1: normalised prices with split shading --------------------------
fig, ax = plt.subplots(figsize=(6, 2.6))
for c in TICKERS:
    ax.plot(close.index, close[c] / close[c].dropna().iloc[0], lw=1,
            label=c, color=PALETTE[c])
for name, (a, b) in SPLITS.items():
    if name in SPLIT_SHADE:
        ax.axvspan(pd.Timestamp(f"{a}-01-01"), pd.Timestamp(f"{b}-12-31"),
                   color=SPLIT_SHADE[name], alpha=0.5, lw=0)
        ax.text(pd.Timestamp(f"{a}-03-01"), ax.get_ylim()[1] * 0.9, name, fontsize=8)
ax.set_yscale("log")
ax.set_ylabel("Growth of $1 (log)")
ax.legend(ncol=4, frameon=False, loc="upper left", bbox_to_anchor=(0, 1.15))
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_prices_splits.png", dpi=200)
plt.show()

# ---- Fig 2: SPY realised vol with the stress threshold --------------------
fig, ax = plt.subplots(figsize=(6, 2.3))
ax.plot(vol20.index, vol20, lw=0.8, color="#1f77b4")
ax.axhline(thr, ls="--", color="#d62728", lw=1, label=f"stress threshold ({thr:.2f})")
ax.fill_between(vol20.index, 0, vol20.max(), where=stress,
                color="#d62728", alpha=0.15, lw=0)
ax.set_ylabel("SPY 20d realized vol\n(annualized)")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_spy_vol_stress.png", dpi=200)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.4  Figures 3-4 of the proposal (calm vs stress corr., ACF)
# ═══════════════════════════════════════════════════════════════════════════
# ---- Fig 3: correlation calm vs stress -----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(6, 2.6))
for ax, m, title in zip(axes, [calm_corr, stress_corr], ["Calm days", "Stress days"]):
    im = ax.imshow(m.values, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(4), TICKERS)
    ax.set_yticks(range(4), TICKERS)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{m.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(title)
fig.colorbar(im, ax=axes, shrink=0.8)
fig.savefig(FIG_DIR / "fig3_corr_calm_vs_stress.png", dpi=200, bbox_inches="tight")
plt.show()

# ---- Fig 4: volatility clustering, ACF of r vs |r| for SPY ---------------
lags = range(1, 31)
fig, ax = plt.subplots(figsize=(6, 2.3))
ax.bar([l - 0.2 for l in lags], [RET["SPY"].autocorr(l) for l in lags],
       width=0.4, label=r"returns $r_t$")
ax.bar([l + 0.2 for l in lags], [RET["SPY"].abs().autocorr(l) for l in lags],
       width=0.4, label=r"|returns| $|r_t|$")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("lag (trading days)")
ax.set_ylabel("SPY autocorrelation")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_spy_acf.png", dpi=200)
plt.show()

---
## 2. Features, windows and chronological splits

**Features** (per ETF, 4 ETFs → `d_x = 16`), all computable at the close of day `t`:

| feature | definition |
|---|---|
| `ret1` | `r_t = log(P_t / P_{t-1})` |
| `vol20` | `std(r_{t-19..t}) · sqrt(252)` |
| `mom20` | `sum(r_{t-19..t})` |
| `rvol` | `log(V_t / mean(V_{t-19..t}))` |

**Target:** `y_{t+1} = r_{t+1}` for all four ETFs (shape 4).

**Leakage rules (proposal §3):**
* a sample belongs to the split of the day it **predicts** (`t+1`), but its window may look back across the boundary — exactly as the proposal specifies;
* windows with missing history are dropped, never filled;
* the standardisation (`μ`, `σ`) and the clipping threshold are fitted on training days only;
* the dynamics-consistency target `E(X_{t+1})` exists only when `t+1` is still inside the same split (training-only signal, never an input).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.1  Causal features -> standardisation -> windows -> splits
# ═══════════════════════════════════════════════════════════════════════════
def build_feature_frame(close, volume, cfg):
    """Features observable at the close of day t (no look-ahead)."""
    tk = list(cfg.tickers)
    r1 = np.log(close[tk]).diff()
    out = {}
    for c in tk:
        out[f"{c}_ret1"] = r1[c]
        out[f"{c}_vol20"] = r1[c].rolling(cfg.vol_win).std() * np.sqrt(252)
        out[f"{c}_mom20"] = r1[c].rolling(cfg.mom_win).sum()
        out[f"{c}_rvol"] = np.log(volume[c] / volume[c].rolling(cfg.rvol_win).mean())
    return out, r1


FEATS, R1 = build_feature_frame(close, volume, CFG)
feat_df = pd.DataFrame(FEATS).replace([np.inf, -np.inf], np.nan)
ret_df = R1[TICKERS]
FEAT_NAMES = list(feat_df.columns)
DATES = feat_df.index
DX, N_OUT, T_TOT = len(FEAT_NAMES), len(TICKERS), len(feat_df)
L = CFG.lookback
print(f"{T_TOT} days x {DX} features -> windows ({L}, {DX}); {N_OUT} targets")
print(FEAT_NAMES)

# ---- standardisation + clipping, fitted on TRAINING days only -------------
dates_t_all = DATES
train_day = ((dates_t_all >= pd.Timestamp(CFG.train[0])) &
             (dates_t_all <= pd.Timestamp(CFG.train[1])))
raw = feat_df.values.astype(np.float64)
mu = np.nanmean(raw[train_day], axis=0)
sd = np.nanstd(raw[train_day], axis=0)
sd = np.where(sd > 0, sd, 1.0)
Z = (raw - mu) / sd
z_clip = float(min(np.nanpercentile(np.abs(Z[train_day]), 99.5), CFG.clip_z))
Z = np.clip(Z, -z_clip, z_clip)
print(f"scaling fitted on {int(train_day.sum())} train days | clip |z| <= {z_clip:.2f}")

# ---- sliding windows; window j ends on day t = L-1+j ----------------------
windows_all = np.lib.stride_tricks.sliding_window_view(Z, (L, DX))[:, 0]
windows_all = windows_all[: T_TOT - L]          # need day t+1 for the target
t_pos = np.arange(L - 1, T_TOT - 1)             # index of predicting day t
y_all = ret_df.values[t_pos + 1].astype(np.float64)   # next-day log returns
dates_t = DATES[t_pos]
dates_tgt = DATES[t_pos + 1]

# ---- split = split of the day that is predicted --------------------------
split_id = np.full(len(t_pos), -1, dtype=int)
for k, name in enumerate(("train", "val", "test")):
    lo = pd.Timestamp(getattr(CFG, name)[0])
    hi = pd.Timestamp(getattr(CFG, name)[1])
    split_id[(dates_tgt >= lo) & (dates_tgt <= hi)] = k
print("candidate windows by target-date split:",
      {n: int((split_id == k).sum()) for k, n in enumerate(("train", "val", "test"))})

# ---- keep windows that are fully observed, labelled, inside a split ------
regime = stress.reindex(DATES).fillna(False).to_numpy(dtype=bool)[t_pos]
labeled = stress_labeled.reindex(DATES).fillna(False).to_numpy(dtype=bool)[t_pos]
ok = (np.isfinite(windows_all).all(axis=(1, 2))
      & np.isfinite(y_all).all(axis=1)
      & (split_id >= 0)
      & labeled)
print(f"kept {int(ok.sum())} / {len(ok)} windows "
      f"(warm-up + unlabelled dropped: {int((~ok).sum())})")

# ---- consistency target E(X_{t+1}) only within the same split ------------
nxt_ok = np.zeros(len(t_pos), dtype=bool)
nxt_ok[:-1] = ok[1:] & (split_id[1:] == split_id[:-1])

keep = np.flatnonzero(ok)
X_win = windows_all[keep].astype(np.float32)
Y_win = y_all[keep].astype(np.float32)
has_next = nxt_ok[keep]
X_next = np.zeros_like(X_win)
X_next[has_next] = windows_all[keep[has_next] + 1].astype(np.float32)
SPLIT = split_id[keep]
REGIME_STRESS = regime[keep]
TGT_DATE = dates_tgt[keep]
SRC_DATE = dates_t[keep]
SPLIT_IDX = {n: np.flatnonzero(SPLIT == k) for k, n in enumerate(("train", "val", "test"))}

print(f"\nX {X_win.shape} | y {Y_win.shape} | X_next {X_next.shape}")
for name in ("train", "val", "test"):
    s = REGIME_STRESS[SPLIT_IDX[name]]
    k = SPLIT_IDX[name]
    print(f"{name:5s} n={len(k):5d} | days t {SRC_DATE[k][0].date()} -> {SRC_DATE[k][-1].date()}"
          f" | targets {TGT_DATE[k][0].date()} -> {TGT_DATE[k][-1].date()}"
          f" | stress={s.mean():6.2%} | n_with_next={int(has_next[k].sum())}")
print("note: the first val/test windows look back into the previous split on purpose "
      "(those days are known at prediction time).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.2  Torch datasets / loaders (index-aligned for regime-stratified metrics)
# ═══════════════════════════════════════════════════════════════════════════
def make_dataset(indices):
    idx = np.asarray(indices, dtype=np.int64)
    return TensorDataset(
        torch.from_numpy(X_win[idx]),
        torch.from_numpy(Y_win[idx]),
        torch.from_numpy(X_next[idx]),
        torch.from_numpy(has_next[idx]),
        torch.from_numpy(idx),
    )


def make_loader(indices, shuffle=False, seed=0, batch_size=None):
    gen = torch.Generator()
    gen.manual_seed(int(seed))
    return DataLoader(make_dataset(indices),
                      batch_size=int(batch_size or CFG.batch_size),
                      shuffle=shuffle, drop_last=False,
                      num_workers=0, generator=gen)


VAL_LOADER = make_loader(SPLIT_IDX["val"])
TEST_LOADER = make_loader(SPLIT_IDX["test"])

_xb, _yb, _xnb, _hnb, _ib = next(iter(VAL_LOADER))
print("batch:", tuple(_xb.shape), tuple(_yb.shape), tuple(_xnb.shape),
      "| has_next", _hnb.dtype, "| idx", _ib[:6].tolist())

---
## 3. Models (proposal §4)

A **shared GRU encoder** `z_t = E_θ(X_t) ∈ R^m` (one layer, 64 units, linear projection, `tanh` so that `|z| < 1`) feeds a model-specific transition; only the transition differs between the three models:

| model | transition | forecast |
|---|---|---|
| `GRU` (direct) | `ẑ_{t+1} = z_t` | `ŷ_{t+1} = h_ψ(ẑ_{t+1})` |
| `GRU+MLP` (dense) | `ẑ_{t+1} = z_t + F_φ(z_t)`, small MLP | `ŷ_{t+1} = h_ψ(ẑ_{t+1})` |
| `GRU+SINDy` (sparse) | `ẑ_{t+1} = z_t + Θ(z_t) Ξ`, `Ξ ∈ R^{p×m}` | `ŷ_{t+1} = h_ψ(ẑ_{t+1})` |

`Θ(z)` holds the constant plus every monomial of total degree `≤ poly_order` (210 terms for `m=4`, degree 6). `h_ψ` is a linear head, shared in shape by all three models. An optional lightweight decoder `z ↦ x_t` supports the proposal's `L_rec` term (`λ_r = 0` disables it).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3.1  Polynomial library Theta(z) for SINDy
# ═══════════════════════════════════════════════════════════════════════════
def poly_exponents(dim, order):
    """Integer exponent vectors of every monomial with total degree <= order."""
    exps = [tuple([0] * dim)]                       # constant term
    for deg in range(1, order + 1):
        for combo in itertools.combinations_with_replacement(range(dim), deg):
            e = [0] * dim
            for i in combo:
                e[i] += 1
            exps.append(tuple(e))
    return exps


def term_label(exp):
    parts = [f"z{i + 1}" + (f"^{k}" if k > 1 else "") for i, k in enumerate(exp) if k > 0]
    return "*".join(parts) if parts else "1"


class PolyLibrary(nn.Module):
    """Theta(z) in R^{B x p}: every monomial of total degree <= order."""

    def __init__(self, dim, order):
        super().__init__()
        exps = poly_exponents(dim, order)
        self.register_buffer("exponents", torch.tensor(exps, dtype=torch.float32))
        self.labels = [term_label(e) for e in exps]
        self.dim, self.order = dim, order

    @property
    def n_terms(self):
        return int(self.exponents.shape[0])

    def forward(self, z):
        # (B, dim) x (p, dim) -> (B, p); z is tanh-bounded so monomials stay O(1)
        return torch.prod(z.unsqueeze(1) ** self.exponents.unsqueeze(0), dim=2)


# ═══════════════════════════════════════════════════════════════════════════
# 3.2  Shared encoder + the three transitions + forecast head
# ═══════════════════════════════════════════════════════════════════════════
class LatentDynamicsGRU(nn.Module):
    """Shared GRU encoder with a pluggable latent transition (direct/dense/sindy)."""

    def __init__(self, n_features, n_out, mode="direct", latent_dim=4, gru_hidden=64,
                 gru_layers=1, head_hidden=0, transition_hidden=32, poly_order=2):
        super().__init__()
        assert mode in ("direct", "dense", "sindy"), mode
        self.mode, self.latent_dim = mode, latent_dim

        # ---- encoder E_theta: X_t -> z_t ----------------------------------
        self.gru = nn.GRU(n_features, gru_hidden, num_layers=gru_layers, batch_first=True)
        self.to_latent = nn.Linear(gru_hidden, latent_dim)

        # ---- transition ----------------------------------------------------
        if mode == "dense":
            self.transition_net = nn.Sequential(
                nn.Linear(latent_dim, transition_hidden), nn.Tanh(),
                nn.Linear(transition_hidden, latent_dim))
            with torch.no_grad():          # start close to the identity map
                self.transition_net[-1].weight.mul_(0.1)
                self.transition_net[-1].bias.zero_()
        elif mode == "sindy":
            self.library = PolyLibrary(latent_dim, poly_order)
            self.Xi = nn.Parameter(torch.zeros(self.library.n_terms, latent_dim))

        # ---- forecast head h_psi -------------------------------------------
        self.head = (nn.Linear(latent_dim, n_out) if head_hidden in (0, None)
                     else nn.Sequential(nn.Linear(latent_dim, head_hidden), nn.Tanh(),
                                        nn.Linear(head_hidden, n_out)))
        # ---- optional lightweight reconstruction (L_rec) --------------------
        self.decoder = nn.Linear(latent_dim, n_features)

    def encode(self, x):
        h, _ = self.gru(x)
        return torch.tanh(self.to_latent(h[:, -1]))

    def transition(self, z):
        if self.mode == "direct":
            return z
        if self.mode == "dense":
            return z + self.transition_net(z)
        return z + self.library(z) @ self.Xi

    def forward(self, x):
        z = self.encode(x)
        z_next = self.transition(z)
        return {"y_hat": self.head(z_next), "z": z, "z_hat": z_next}


# ---- shape / parameter sanity check ---------------------------------------
_probe = next(iter(VAL_LOADER))[0][:8].to(DEVICE)
for _mode in ("direct", "dense", "sindy"):
    _m = LatentDynamicsGRU(DX, N_OUT, mode=_mode, latent_dim=CFG.latent_dim,
                           gru_hidden=CFG.gru_hidden, head_hidden=CFG.head_hidden,
                           transition_hidden=CFG.transition_hidden,
                           poly_order=CFG.poly_order).to(DEVICE)
    _out = _m(_probe)
    print(f"{_mode:7s} y_hat{tuple(_out['y_hat'].shape)} z{tuple(_out['z'].shape)} "
          f"z_hat{tuple(_out['z_hat'].shape)} | params={sum(p.numel() for p in _m.parameters()):,}"
          + (f" | Xi={tuple(_m.Xi.shape)}" if _mode == "sindy" else ""))
print("library terms:", _m.library.labels[:6], "...", _m.library.labels[-3:])

---
## 4. Objective and training function

The sparse model minimises

$$\mathcal{L} = \underbrace{\mathrm{MSE}(\hat y_{t+1}, y_{t+1})}_{\mathcal{L}_{pred}} + \lambda_d\,\big\|\,\mathrm{sg}\big(E_\theta(X_{t+1})\big) - \hat z_{t+1}\big\|_2^2 + \lambda_s \|\Xi\|_1 + \lambda_r \mathcal{L}_{rec}$$

* `sg(·)` = stop-gradient: the next window is a **target** (`CFG.detach_next`), and it is only used where `t+1` lies in the same split;
* `GRU+MLP` keeps the consistency term but has no `‖Ξ‖₁`; `GRU` has neither transition term;
* optimiser AdamW, gradient clipping, early stopping on **validation MSE** with restoration of the best weights (proposal §4).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4.1  Loss (proposal eq. 3), prediction, training loop
# ═══════════════════════════════════════════════════════════════════════════
def batch_loss(model, batch, lam_d=0.0, lam_s=0.0, lam_r=0.0, detach_next=True):
    """One minibatch of L_pred + lam_d*L_cons + lam_s*|Xi|_1 + lam_r*L_rec."""
    x, y, x_next, has_next, _ = batch
    out = model(x)
    l_pred = Fn.mse_loss(out["y_hat"], y)
    total, logs = l_pred, {"pred": float(l_pred.detach())}

    if model.mode != "direct":
        ctx = torch.no_grad if detach_next else torch.enable_grad
        with ctx():
            z_next = model.encode(x_next)          # shared encoder, target side
        if bool(has_next.any()):
            l_cons = (out["z_hat"][has_next] - z_next[has_next]).pow(2).sum(dim=1).mean()
            total = total + lam_d * l_cons
            logs["cons"] = float(l_cons.detach())

    if model.mode == "sindy" and lam_s > 0:
        # per-coefficient mean |Xi| so that lambda_s = 1 is a meaningful weight
        l_l1 = model.Xi.abs().sum() / model.Xi.numel()
        total = total + lam_s * l_l1
        logs["l1"] = float(l_l1.detach())

    if lam_r > 0:
        l_rec = Fn.mse_loss(model.decoder(out["z"]), x[:, -1, :])
        total = total + lam_r * l_rec
        logs["rec"] = float(l_rec.detach())

    return total, logs


@torch.no_grad()
def predict(model, loader):
    """Return (predictions, targets, dataset indices) in original return units."""
    model.eval()
    preds, trues, idxs = [], [], []
    for x, y, _xn, _hn, idx in loader:
        preds.append(model(x.to(DEVICE))["y_hat"].cpu().numpy().astype(np.float64))
        trues.append(y.numpy().astype(np.float64))
        idxs.append(idx.numpy())
    return np.concatenate(preds), np.concatenate(trues), np.concatenate(idxs)


def mse_of(model, loader):
    p, t, _ = predict(model, loader)
    return float(np.mean((p - t) ** 2))


def fit_model(mode, lam_d, lam_s, lam_r=0.0, seed=0, cfg=CFG, verbose=False):
    """Train one model; early stopping on validation MSE, best weights restored."""
    set_seed(seed)
    model = LatentDynamicsGRU(
        n_features=DX, n_out=N_OUT, mode=mode, latent_dim=cfg.latent_dim,
        gru_hidden=cfg.gru_hidden, gru_layers=cfg.gru_layers,
        head_hidden=cfg.head_hidden, transition_hidden=cfg.transition_hidden,
        poly_order=cfg.poly_order).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    train_loader = make_loader(SPLIT_IDX["train"], shuffle=True, seed=seed)

    best_val, best_epoch = float("inf"), 0
    best_state = copy.deepcopy(model.state_dict())
    bad, hist = 0, []
    bar = tqdm(range(1, cfg.epochs + 1), leave=False, disable=not verbose,
               desc=f"{mode} ld={lam_d} ls={lam_s} s={seed}")
    for epoch in bar:
        model.train()
        tr_loss, nb = 0.0, 0
        for batch in train_loader:
            batch = [b.to(DEVICE) for b in batch]
            opt.zero_grad(set_to_none=True)
            loss, _ = batch_loss(model, batch, lam_d, lam_s, lam_r, cfg.detach_next)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            tr_loss += float(loss.detach())
            nb += 1
        val_mse = mse_of(model, VAL_LOADER)
        hist.append(dict(epoch=epoch, train_loss=tr_loss / max(nb, 1), val_mse=val_mse))
        if verbose:
            bar.set_postfix(train=f"{tr_loss / max(nb, 1):.5f}", val=f"{val_mse:.6f}")
        if val_mse < best_val:
            best_val, best_epoch, bad = val_mse, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= cfg.patience:
                if verbose:
                    print(f"  early stop @ epoch {epoch} (best {best_epoch})")
                break
    model.load_state_dict(best_state)
    return model, dict(history=hist, best_epoch=best_epoch, best_val_mse=best_val)

---
## 5. Training the three models

No hyper-parameter search: there is nothing to tune, because `λ_d` and `λ_s` are switches (see 0.2). The three rows below are the whole experimental design — the models share the encoder, the head, the features, the splits, the optimiser and the stopping rule, and differ only in the transition:

| model | transition | `λ_d` | `λ_s` |
|---|---|---|---|
| `GRU` | `ẑ_{t+1} = z_t` | 0 | 0 |
| `GRU+MLP` | `ẑ_{t+1} = z_t + F_φ(z_t)` | 1 | 0 |
| `GRU+SINDy` | `ẑ_{t+1} = z_t + Θ(z_t)Ξ` | 1 | 1 |

Each model is trained with seeds `0, 1, 2` so that the reported numbers carry a seed-to-seed spread (proposal §5). The test split is not touched here.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5.1  Train the three models (3 seeds each)
# ═══════════════════════════════════════════════════════════════════════════
RUNS, t_start = [], time.time()
for label in MODELS:
    lam_d = LAMBDAS[label]["lambda_d"]
    lam_s = LAMBDAS[label]["lambda_s"]
    for seed in CFG.seeds:
        model, info = fit_model(MODE_OF[label], lam_d, lam_s, CFG.lambda_r,
                                seed, verbose=True)
        RUNS.append(dict(model=label, mode=MODE_OF[label], seed=seed,
                         lambda_d=lam_d, lambda_s=lam_s,
                         model_obj=model, info=info))
        print(f"{label:9s} seed={seed} | lambda_d={lam_d:.0f} lambda_s={lam_s:.0f}"
              f" | val MSE {info['best_val_mse']:.4e}"
              f" | best epoch {info['best_epoch']:3d} of {len(info['history']):3d}")
print(f"\n{len(RUNS)} runs finished in {(time.time() - t_start) / 60:.1f} min")

---
## 6. Results

Metrics, all in next-day log-return space:

* **MSE ×10⁴**, **MAE (%)**, **directional accuracy (%)**, where a non-negative forecast counts as "up" — so the zero-return predictor is scored as always predicting "up", and its directional accuracy equals the share of positive returns;
* reported for **all** days, **calm** days and **stressed** days, where the stratum is the stress label of the day the forecast is made (proposal §3 — strata are used for evaluation only, never as inputs);
* the `zero-return` predictor is the proposal's financial benchmark.

### Validation split (2020–2021)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6.1  Metric helpers + validation table
# ═══════════════════════════════════════════════════════════════════════════
def metrics(pred, true):
    err = pred - true
    return dict(mse=float(np.mean(err ** 2)),
                mae=float(np.mean(np.abs(err))),
                da=float(np.mean(np.where(pred >= 0, 1, -1) == np.where(true >= 0, 1, -1))))


def metrics_by_regime(pred, true, stress_mask):
    out = {"all": metrics(pred, true)}
    if stress_mask.any():
        out["stress"] = metrics(pred[stress_mask], true[stress_mask])
    if (~stress_mask).any():
        out["calm"] = metrics(pred[~stress_mask], true[~stress_mask])
    return out


_PRED_CACHE: Dict[Tuple[int, str], tuple] = {}


def pred_of(run, which):
    """Predictions (val or test) of one trained run, cached."""
    key = (id(run), which)
    if key not in _PRED_CACHE:
        loader = VAL_LOADER if which == "val" else TEST_LOADER
        _PRED_CACHE[key] = predict(run["model_obj"], loader)
    return _PRED_CACHE[key]


def runs_to_frame(runs, which):
    rows = []
    for r in runs:
        pred, true, idx = pred_of(r, which)
        row = dict(model=r["model"], seed=r["seed"])
        for part, d in metrics_by_regime(pred, true, REGIME_STRESS[idx]).items():
            row.update({f"{part}_{k}": v for k, v in d.items()})
        rows.append(row)
    return pd.DataFrame(rows)


def baseline_frame(which):
    idx = SPLIT_IDX[which]
    true = Y_win[idx].astype(np.float64)
    row = dict(model="zero-return", seed=-1)
    for part, d in metrics_by_regime(np.zeros_like(true), true,
                                     REGIME_STRESS[idx]).items():
        row.update({f"{part}_{k}": v for k, v in d.items()})
    return pd.DataFrame([row])


SPEC = [("all_mse", 1e4, "MSE x1e4"),
        ("calm_mse", 1e4, "calm MSE x1e4"),
        ("stress_mse", 1e4, "stress MSE x1e4"),
        ("all_mae", 1e2, "MAE (%)"),
        ("all_da", 1e2, "dir.acc (%)")]


def report_table(frame, spec=SPEC, row_order=None):
    grp = frame.groupby("model")
    cols, index = {}, None
    for col, scale, name in spec:
        mean = grp[col].mean() * scale
        sd = grp[col].std(ddof=0).fillna(0.0) * scale
        index = mean.index
        cols[name] = [f"{m:8.3f} +- {s:6.3f}" for m, s in zip(mean, sd)]
    out = pd.DataFrame(cols, index=index)
    if row_order:
        out = out.reindex([r for r in row_order if r in out.index])
    return out


VAL_FRAME = pd.concat([runs_to_frame(RUNS, "val"), baseline_frame("val")],
                      ignore_index=True)
print("validation MSE, calm/stress strata, MAE and directional accuracy")
print("(mean +- std over seeds; the baseline has a single deterministic prediction)\n")
display(report_table(VAL_FRAME, row_order=MODELS + ["zero-return"]))

### Held-out test split (2022–2025)

> Per the proposal, the test split is inspected only now, once the design is frozen (the `λ` switches of section 5 are fixed, nothing was tuned on test). This is a single look — do not iterate on the test numbers.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6.2  Held-out test table  (+ per-asset breakdown)
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 78)
print("HELD-OUT TEST RESULTS (2022-2025). Design fixed in sections 0-5.")
print("=" * 78)

TEST_FRAME = pd.concat([runs_to_frame(RUNS, "test"), baseline_frame("test")],
                       ignore_index=True)
display(report_table(TEST_FRAME, row_order=MODELS + ["zero-return"]))

# ---- per-asset test MSE ---------------------------------------------------
rows = []
for label in MODELS:
    per_seed = [((pred_of(r, "test")[0] - pred_of(r, "test")[1]) ** 2).mean(axis=0)
                for r in RUNS if r["model"] == label]
    row = dict(model=label)
    row.update({t: float(v) for t, v in zip(TICKERS, np.mean(per_seed, axis=0) * 1e4)})
    rows.append(row)
true_test = Y_win[SPLIT_IDX["test"]].astype(np.float64)
rows.append(dict(model="zero-return",
                 **{t: float(v) for t, v in zip(TICKERS, (true_test ** 2).mean(axis=0) * 1e4)}))
print("\nper-asset test MSE x1e4:")
display(pd.DataFrame(rows).set_index("model").round(4))

# ---- compare each model with the zero-return benchmark --------------------
zero_mse = float((true_test ** 2).mean())
best_mse = TEST_FRAME[TEST_FRAME.model != "zero-return"].groupby("model").all_mse.mean()
print(f"\nzero-return test MSE = {zero_mse:.4e}")
for k, v in best_mse.items():
    print(f"{k:9s} test MSE = {v:.4e}  ({'better' if v < zero_mse else 'WORSE'} than zero-return)")

---
## 7. SINDy dynamics diagnostics (proposal §5)

1. **Retained terms** — coefficients of `Ξ` whose magnitude exceeds 1 % of the largest one, printed per latent coordinate.
2. **One-step latent transition error** — `mean_i ‖E(X_{t+1}) − ẑ_{t+1}‖²` on validation.
3. **Term stability in a frozen coordinate system** — the encoder and head of the best `GRU+SINDy` run are frozen and only `Ξ` is re-fitted on three time blocks; agreement between active sets is reported as the Jaccard index.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7.1  Retained terms, transition error, frozen-encoder stability
# ═══════════════════════════════════════════════════════════════════════════
sindy_runs = sorted([r for r in RUNS if r["model"] == "GRU+SINDy"],
                    key=lambda r: r["info"]["best_val_mse"])
best_sindy = sindy_runs[0]
sindy_model = best_sindy["model_obj"]
labels = sindy_model.library.labels
Xi = sindy_model.Xi.detach().cpu().numpy()
tol = 1e-2 * float(np.abs(Xi).max())
active = np.abs(Xi) > tol

print(f"seed {best_sindy['seed']} | library: {len(labels)} terms, degree <= "
      f"{sindy_model.library.order} | active entries {int(active.sum())}/{active.size}")
print("\nlearned latent dynamics (terms above 1% of max |Xi|):")
for j in range(Xi.shape[1]):
    order = np.argsort(-np.abs(Xi[:, j]))[:6]
    terms = [f"{Xi[i, j]:+.3e}*{labels[i]}" for i in order if abs(Xi[i, j]) > tol]
    print(f"  dz{j + 1} = " + (" + ".join(terms) if terms else "0"))

term_rows = [dict(latent=f"z{j + 1}", term=labels[i], coef=float(Xi[i, j]),
                  active=bool(active[i, j]))
             for i in range(len(labels)) for j in range(Xi.shape[1])]
TERMS_DF = pd.DataFrame(term_rows)
TERMS_DF.sort_values("coef", key=np.abs, ascending=False).to_csv(
    RES_DIR / "sindy_terms.csv", index=False)
print(f"\nactive terms per latent coordinate: "
      f"{ {f'z{j + 1}': int(active[:, j].sum()) for j in range(Xi.shape[1])} }")

# ---- one-step latent transition error ------------------------------------
@torch.no_grad()
def latent_transition_error(model, loader):
    model.eval()
    errs = []
    for x, _y, x_next, has_next, _idx in loader:
        hn = has_next.to(DEVICE)
        out = model(x.to(DEVICE))
        z_next = model.encode(x_next.to(DEVICE))
        if bool(hn.any()):
            errs.append((out["z_hat"][hn] - z_next[hn]).pow(2).sum(1).cpu())
    e = torch.cat(errs)
    return float(e.mean()), float(e.median())


for name, loader in (("val", VAL_LOADER), ("test", TEST_LOADER)):
    mean_e, med_e = latent_transition_error(sindy_model, loader)
    print(f"one-step latent transition error [{name}]: mean {mean_e:.5f} | median {med_e:.5f}")

# ---- term stability with the encoder held fixed --------------------------
def refit_xi(model, sel, lam_s, epochs=40, lr=3e-3):
    """Freeze every parameter except Xi and re-fit the transition on one block."""
    model.train()
    for p in model.parameters():
        p.requires_grad_(False)
    model.Xi.requires_grad_(True)
    opt = torch.optim.Adam([model.Xi], lr=lr)
    loader = make_loader(sel, shuffle=True, seed=0)
    for _ in range(epochs):
        for batch in loader:
            batch = [b.to(DEVICE) for b in batch]
            opt.zero_grad(set_to_none=True)
            loss, _ = batch_loss(model, batch, 0.0, lam_s, 0.0, True)
            loss.backward()
            opt.step()
    out = model.Xi.detach().cpu().numpy().copy()
    for p in model.parameters():
        p.requires_grad_(True)
    return out


T = pd.Series(TGT_DATE)
blocks = {
    "train 2010-2014": np.flatnonzero((SPLIT == 0) & (T <= pd.Timestamp("2014-12-31")).values),
    "train 2015-2019": np.flatnonzero((SPLIT == 0) & (T >= pd.Timestamp("2015-01-01")).values),
    "val 2020-2021":   np.flatnonzero(SPLIT == 1),
}
lam_s_best = LAMBDAS["GRU+SINDy"]["lambda_s"]
FROZEN_XI, FROZEN_MASK = {}, {}
for name, sel in blocks.items():
    m = copy.deepcopy(sindy_model)
    xi_k = refit_xi(m, sel, lam_s_best)
    FROZEN_XI[name] = xi_k
    mask_k = np.abs(xi_k) > 1e-2 * float(np.abs(xi_k).max())
    FROZEN_MASK[name] = mask_k
    print(f"  block {name:16s} n={len(sel):4d} | active {int(mask_k.sum()):3d}")

print("\nactive-set agreement across blocks (Jaccard, frozen encoder):")
names = list(blocks)
for a, b in itertools.combinations(names, 2):
    ma, mb = FROZEN_MASK[a], FROZEN_MASK[b]
    jac = float((ma & mb).sum() / max((ma | mb).sum(), 1))
    print(f"  {a:16s} vs {b:16s}: {jac:.3f}")

---
## 8. Figures

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 8.1  Validation curves and error by model
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))

ax = axes[0]
for label in MODELS:
    hist = [r for r in RUNS
            if r["model"] == label and r["seed"] == CFG.seeds[0]][0]["info"]["history"]
    ax.plot([h["epoch"] for h in hist], [h["val_mse"] * 1e4 for h in hist],
            label=label, color=COLORS[label], lw=1.4)
ax.set_xlabel("epoch")
ax.set_ylabel("validation MSE x1e4")
ax.set_title("Validation MSE during training (seed 0)")
ax.legend(frameon=False)

ax = axes[1]
width, xs = 0.26, np.arange(len(MODELS))
for k, (name, frame) in enumerate((("validation", VAL_FRAME), ("test", TEST_FRAME))):
    vals = [frame[frame.model == m].all_mse.mean() * 1e4 for m in MODELS]
    sd = [frame[frame.model == m].all_mse.std(ddof=0) * 1e4 for m in MODELS]
    ax.bar(xs + (k - 0.5) * width, vals, width, yerr=sd, capsize=3,
           label=name, color=["#7fb2d9", "#d98b7f"][k])
zero_val = VAL_FRAME[VAL_FRAME.model == "zero-return"].all_mse.mean() * 1e4
ax.axhline(zero_val, ls="--", lw=1, color="gray", label="zero-return")
ax.set_xticks(xs, MODELS)
ax.set_ylabel("MSE x1e4")
ax.set_title("Next-day return MSE (mean over seeds)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig5_training_and_mse.png", dpi=200)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 8.2  Forecast scatter, calm/stress strata, SINDy coefficients
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

# ---- (a) test-set scatter for SPY, one seed per model ---------------------
ax = axes[0]
spy = TICKERS.index("SPY")
for label in MODELS:
    r = [r for r in RUNS if r["model"] == label and r["seed"] == CFG.seeds[0]][0]
    p, t, _ = pred_of(r, "test")
    ax.scatter(t[:, spy] * 100, p[:, spy] * 100, s=6, alpha=0.45,
               color=COLORS[label], label=label)
lim = np.nanmax(np.abs(true_test[:, spy])) * 100
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8)
ax.set_xlabel("actual SPY return (%)")
ax.set_ylabel("predicted SPY return (%)")
ax.set_title("Test forecasts (SPY, seed 0)")
ax.legend(frameon=False, markerscale=2)

# ---- (b) calm vs stress MSE on test --------------------------------------
ax = axes[1]
width, xs = 0.26, np.arange(len(MODELS))
for k, part in enumerate(("calm", "stress")):
    vals = [TEST_FRAME[TEST_FRAME.model == m][f"{part}_mse"].mean() * 1e4 for m in MODELS]
    ax.bar(xs + (k - 0.5) * width, vals, width, label=part,
           color=["#8fbf8f", "#c96f6f"][k])
ax.set_xticks(xs, MODELS)
ax.set_ylabel("test MSE x1e4")
ax.set_title("Calm vs stressed days (test)")
ax.legend(frameon=False)

# ---- (c) largest SINDy coefficients --------------------------------------
ax = axes[2]
top = TERMS_DF.reindex(TERMS_DF.coef.abs().sort_values(ascending=False).index)
top_terms = list(dict.fromkeys(top.term))[:18]
mat = np.zeros((len(top_terms), Xi.shape[1]))
for i, name in enumerate(top_terms):
    sub = TERMS_DF[TERMS_DF.term == name]
    for j in range(Xi.shape[1]):
        mat[i, j] = float(sub[sub.latent == f"z{j + 1}"].coef.iloc[0])
vmax = float(np.abs(mat).max()) or 1.0
im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_yticks(range(len(top_terms)), top_terms, fontsize=6)
ax.set_xticks(range(Xi.shape[1]), [f"z{j + 1}" for j in range(Xi.shape[1])])
ax.set_title(r"learned $\Xi$ (largest terms)")
fig.colorbar(im, ax=ax, shrink=0.85)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig6_forecasts_strata_sindy.png", dpi=200)
plt.show()

---
## 9. Save artefacts and summary

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 9.1  Persist metrics / config and print a summary
# ═══════════════════════════════════════════════════════════════════════════
REPORT = report_table(VAL_FRAME, row_order=MODELS + ["zero-return"])
REPORT_TEST = report_table(TEST_FRAME, row_order=MODELS + ["zero-return"])
REPORT.to_csv(RES_DIR / "metrics_validation.csv")
REPORT_TEST.to_csv(RES_DIR / "metrics_test.csv")
pd.DataFrame([dict(model=r["model"], seed=r["seed"], lambda_d=r["lambda_d"],
                   lambda_s=r["lambda_s"], best_epoch=r["info"]["best_epoch"],
                   val_mse=r["info"]["best_val_mse"]) for r in RUNS]
             ).to_csv(RES_DIR / "runs.csv", index=False)
with open(RES_DIR / "config.json", "w", encoding="utf-8") as fh:
    json.dump({"config": asdict(CFG), "lambdas": LAMBDAS,
               "sindy_library_terms": int(len(labels)),
               "sindy_active_entries": int(active.sum())}, fh, indent=2, default=str)

summary = [
    "=" * 78,
    "SUMMARY",
    "=" * 78,
    f"panel: {TICKERS} | {close.index[0].date()} .. {close.index[-1].date()}",
    f"windows: L={L} | features d_x={DX} | targets N={N_OUT}",
    f"samples: train {len(SPLIT_IDX['train'])} | val {len(SPLIT_IDX['val'])} | "
    f"test {len(SPLIT_IDX['test'])}",
    f"stress threshold: {thr:.3f} (train 80th pct) | stress share: "
    f"train {REGIME_STRESS[SPLIT_IDX['train']].mean():.1%}, "
    f"val {REGIME_STRESS[SPLIT_IDX['val']].mean():.1%}, "
    f"test {REGIME_STRESS[SPLIT_IDX['test']].mean():.1%}",
    f"SINDy library: {len(labels)} terms (degree <= {CFG.poly_order}) | "
    f"active {int(active.sum())}/{active.size}",
    "",
    "model      val MSE x1e4    test MSE x1e4   test dir.acc",
]
for m in MODELS + ["zero-return"]:
    v = VAL_FRAME[VAL_FRAME.model == m].all_mse.mean() * 1e4
    t = TEST_FRAME[TEST_FRAME.model == m].all_mse.mean() * 1e4
    d = TEST_FRAME[TEST_FRAME.model == m].all_da.mean() * 100
    summary.append(f"{m:10s} {v:10.3f} {t:14.3f} {d:12.2f}%")
summary += ["", f"artefacts: {RES_DIR}/, {FIG_DIR}/"]
print("\n".join(summary))

---
## 10. Notes, caveats and next steps

**Reading the numbers.** Next-day ETF returns are close to unpredictable, so differences of a few percent in MSE between models should be read against the seed-to-seed spread shown in every table, not as a ranking on their own. The zero-return row is the honest reference point: any model that does not beat it out of sample is not adding forecast value, whatever its in-sample behaviour. The proposal's claim is about a *controlled comparison of an inductive bias*, and that is what these tables support.

**What was deliberately kept simple / configurable.**
* `λ_d` and `λ_s` are on/off switches, not tuned weights; the §5 table is the whole experimental design.
* the ℓ1 term is the **mean** absolute coefficient, so `λ_s = 1` has the same order of magnitude as a normalised MSE. The proposal's raw `Σ|Ξ|` is the same line without the `/ model.Xi.numel()` division — with that version `λ_s` would have to be ~1e-3 instead of 1.
* `poly_order = 6` gives a 210-term library; the proposal's figure describes the 15-term quadratic case. Set `CFG.poly_order = 2` to reproduce it exactly (cheaper and easier to read the discovered equations).
* `λ_r = 0` disables the optional reconstruction loss; set `CFG.lambda_r > 0` to switch it on (the decoder exists).
* `CFG.detach_next = False` lets gradients flow into the consistency target as written in eq. (3) verbatim.
* Feature definitions (`vol20`, `mom20`, `rvol`) follow the proposal's candidate list; the proposal leaves the exact lookbacks to training/validation data, and they are all config fields.

**Leakage checklist (all verified in section 2).**
* scaling and clipping statistics come from 2010–2019 days only;
* a sample's split is decided by the day it predicts, so validation/test windows may legitimately look back into the previous split;
* future information enters only as a *training target* (`y_{t+1}`, and `E(X_{t+1})` where `t+1` is inside the same split);
* the test split is evaluated once, after all choices are frozen.

**Natural extensions** (explicitly secondary in the proposal): a rolling-origin refit, a transaction-cost-aware backtest, and regime-conditioned coefficient matrices `Ξ_k`, for which the `sindy` transition is the natural place to start.